In [ ]:
from flowrec.data.vk_lsvd import VKLSVDDataConfig, VKLSVDDataModule



In [ ]:
cfg = VKLSVDDataConfig(
    use_hf=True,
    split_strategy="provided",
    min_user_interactions=5,
    max_history_len=50,
    batch_size=256,
    train_all_prefixes=False,
    subsample_name='ur0.01_ir0.01',
    global_eval_target_mode='last_per_user',
    rating_mode='timespent',
    min_timespent=1.0
)

dm = VKLSVDDataModule(cfg)
dm.setup()

print(dm.summary())
batch = next(iter(dm.train_dataloader()))

In [ ]:
from flowrec.item_representation.vk_lsvd import (
    VKLSVDItemEmbeddingConfig,
    VKLSVDItemEmbeddingBuilder,
)

store = VKLSVDItemEmbeddingBuilder(
    VKLSVDItemEmbeddingConfig(
        embeddings_path="../data/raw/VK-LSVD/item_embeddings.npz",
        embedding_dim=64,
    )
).build()

store.save("../artifacts/item_embeddings/vk_lsvd_embeddings.pt")

In [ ]:
from flowrec.item_representation.semantic_id import (
    RQVAEConfig,
    RQVAETrainConfig,
    RQVAEItemTokenizerConfig,
    RQVAEItemTokenizer,
)

tokenizer = RQVAEItemTokenizer(
    RQVAEItemTokenizerConfig(
        rqvae=RQVAEConfig(
            input_dim=store.embeddings.shape[1],
            latent_dim=32,
            num_codebooks=3,
            codebook_size=256,
        ),
        train=RQVAETrainConfig(epochs=50, device="cpu"),
        normalize="l2",
        collision_policy="append",
    )
)

tokenizer.fit(store.embeddings, store.item_ids)
tokenizer.save("../artifacts/tokenizers/vk_lsvd_rqvae")

In [ ]:
cfg = VKLSVDDataConfig(
    semantic_id_path="../artifacts/tokenizers/vk_lsvd_rqvae/semantic_ids.json",
    use_hf=True,
    split_strategy="provided",
    min_user_interactions=5,
    max_history_len=50,
    batch_size=256,
    train_all_prefixes=False,
    subsample_name='ur0.01_ir0.01',
    global_eval_target_mode='last_per_user',
    rating_mode='timespent',
    min_timespent=1.0
)

dm = VKLSVDDataModule(cfg)
dm.setup()

batch = next(iter(dm.train_dataloader()))
print(batch["history_token_ids"].shape)
print(batch["target_token_ids"].shape)